# Lesson 12 — Training, Validation, and Test Sets

**What this notebook does:** it shows, hands-on, *why* we hide data from a model and how the pros split their examples into **three** piles instead of two.

First we commit the classic beginner mistake on purpose — a model that **memorises** its training tickets and scores a perfect 1.00 on them, then collapses on new tickets. That is the **cardinal sin** of testing on your training data.

Then we do it properly: split into **train / validation / test**, use the validation pile to *choose* between two candidate models, and only touch the test pile once, at the very end, as the honest final judge.

Every code cell has short comments on each line, so you can read the code straight through. The markdown before each cell explains the bigger *why*.

## Stage 1 — The data

We use 18 labelled support tickets across three categories (**shipping**, **billing**, **account**), arranged so the split is clean:

- rows **0–11** → the **training** set (12 tickets, 4 per category)
- rows **12–14** → the **validation** set (3 tickets, 1 per category)
- rows **15–17** → the **test** set (3 tickets, 1 per category)

As in Lesson 11, each ticket is paired with its true category, and we chop each one into lowercase words (**tokens**) so the computer has countable pieces to work with.

In [1]:
import pandas as pd  # our table tool (Lesson 06)

# 18 tickets = (text, category). First 12 rows train, next 3 validate, last 3 test.
tickets = pd.DataFrame([
    # --- training rows 0..11 (4 shipping, 4 billing, 4 account) ---
    ("where is my package it has not arrived yet", "shipping"),
    ("my delivery is late and tracking has not updated", "shipping"),
    ("when will my order ship out", "shipping"),
    ("my package still has not arrived and the shipping is slow", "shipping"),
    ("i was charged twice for my order please refund", "billing"),
    ("there is a wrong charge on my card", "billing"),
    ("i need a refund for a payment i did not make", "billing"),
    ("why was i charged twice on my invoice", "billing"),
    ("i cannot log in to my account", "account"),
    ("how do i reset my password", "account"),
    ("my login is not working and i am locked out", "account"),
    ("i forgot my password and cannot log in to my account", "account"),
    # --- validation rows 12..14 (1 per category) ---
    ("i think my order is late", "shipping"),
    ("there is a wrong charge on my card again", "billing"),
    ("i cannot log in and reset my password", "account"),
    # --- test rows 15..17 (1 per category) ---
    ("my package has not arrived and delivery is late", "shipping"),
    ("i was charged twice and need a refund", "billing"),
    ("i forgot my password and cannot log in", "account"),
], columns=["text", "category"])  # name the two columns

# Chop every ticket into a list of lowercase words, saved in a new "tokens" column
tickets["tokens"] = tickets["text"].str.lower().str.split()  # .str applies to the whole column

print("Total tickets:", len(tickets))  # expect 18
tickets[["text", "category"]]           # show the tickets and their true categories

Total tickets: 18


,text,category
0,where is my package it has not arrived yet,shipping
1,my delivery is late and tracking has not updated,shipping
2,when will my order ship out,shipping
3,my package still has not arrived and the shipp...,shipping
4,i was charged twice for my order please refund,billing
5,there is a wrong charge on my card,billing
6,i need a refund for a payment i did not make,billing
7,why was i charged twice on my invoice,billing
8,i cannot log in to my account,account
9,how do i reset my password,account


## The cardinal sin — a model that just memorises

Before doing it right, let's do it wrong on purpose, so the mistake is unforgettable.

We build a **memoriser**: it simply stores a lookup table of `exact ticket text → its category`. It "learns" nothing about *why* a ticket is billing — it just remembers the answer key. For any ticket it has never seen, it has no idea, so it falls back to guessing the most common category.

Watch what happens when we grade it on the **same tickets it memorised** versus on **new tickets**.

In [2]:
train = tickets.iloc[:12]         # rows 0..11  -> the model may study these
validation = tickets.iloc[12:15]  # rows 12..14 -> for choosing between models
test = tickets.iloc[15:]          # rows 15..17 -> locked away, final judge only

# The memoriser: remember every training ticket's exact text -> category
lookup = {}                        # exact-text answer key, built by hand
for _, row in train.iterrows():    # go through training tickets one by one
    lookup[row["text"]] = row["category"]  # remember this exact ticket's true answer
    print(f"memorised: '{row['text']}' -> {row['category']}")

print(f"\nlookup table has {len(lookup)} entries\n")

# Find the most common training category, to use as a fallback guess
category_counts = {}                   # category -> how many training tickets have it
for _, row in train.iterrows():        # go through training tickets again
    if row["category"] not in category_counts:  # first time seeing this category?
        category_counts[row["category"]] = 0      # start its count at zero
    category_counts[row["category"]] += 1        # add one for this ticket

print("category counts in training data:", category_counts, "\n")

fallback = None       # will hold the most common category
highest_count = -1    # lower than any real count, so the first category always wins at first
for category, count in category_counts.items():  # check each category's count
    print(f"checking '{category}': count = {count}, best so far = {fallback} ({highest_count})")
    if count > highest_count:                       # is this the new most common one?
        highest_count = count
        fallback = category
        print(f"    -> new best: {fallback}")

print(f"\nfallback category = {fallback}\n")

def memoriser_predict(text):
    if text in lookup:      # have we seen this exact ticket text before?
        print(f"  '{text}' was memorised -> {lookup[text]}")
        return lookup[text]   # yes -> recall its stored answer
    print(f"  '{text}' was NEVER seen during training -> guessing fallback '{fallback}'")
    return fallback           # no -> just guess the most common training category

# Accuracy = fraction of a table's tickets labelled correctly
def memoriser_accuracy(df):
    correct = 0                             # how many guesses were right so far
    for _, row in df.iterrows():            # check each ticket one by one
        guess = memoriser_predict(row["text"])  # the memoriser's guess for this ticket
        is_correct = guess == row["category"]     # compare guess to the true answer
        print(f"  guess = {guess:<10} true = {row['category']:<10} correct = {is_correct}")
        if is_correct:
            correct += 1                              # count it if right
    print(f"  {correct} / {len(df)} correct\n")
    return correct / len(df)

print("=== scoring memoriser on its OWN TRAINING tickets ===")
train_acc = memoriser_accuracy(train)
print(f"Memoriser on its OWN training tickets: {train_acc:.2f}\n")  # looks perfect

print("=== scoring memoriser on NEW TEST tickets ===")
test_acc = memoriser_accuracy(test)
print(f"Memoriser on NEW held-out test tickets: {test_acc:.2f}")  # the truth

memorised: 'where is my package it has not arrived yet' -> shipping
memorised: 'my delivery is late and tracking has not updated' -> shipping
memorised: 'when will my order ship out' -> shipping
memorised: 'my package still has not arrived and the shipping is slow' -> shipping
memorised: 'i was charged twice for my order please refund' -> billing
memorised: 'there is a wrong charge on my card' -> billing
memorised: 'i need a refund for a payment i did not make' -> billing
memorised: 'why was i charged twice on my invoice' -> billing
memorised: 'i cannot log in to my account' -> account
memorised: 'how do i reset my password' -> account
memorised: 'my login is not working and i am locked out' -> account
memorised: 'i forgot my password and cannot log in to my account' -> account

lookup table has 12 entries

category counts in training data: {'shipping': 4, 'billing': 4, 'account': 4} 

checking 'shipping': count = 4, best so far = None (-1)
    -> new best: shipping
checking 'billing':

**That gap is the whole lesson.** A perfect 1.00 on the training tickets meant *nothing* — the model had simply seen the answers. The honest score, on tickets it never saw, is barely better than random guessing. If we had only ever checked the training score, we'd have shipped a useless model believing it was flawless.

So from here on we never grade a model on data it was allowed to study.

## Doing it right — three piles, three jobs

We already sliced the data into three sets above. Their jobs:

- **train** (12 tickets) — the model learns its numbers from *only* these.
- **validation** (3 tickets) — *we* use these to **choose** between options during development. Never learned from.
- **test** (3 tickets) — the final, honest judge. Touched exactly **once**, at the very end.

Let's confirm the sizes.

In [3]:
print("Training tickets:  ", len(train))       # expect 12
print("Validation tickets:", len(validation))  # expect 3
print("Test tickets:      ", len(test))        # expect 3

Training tickets:   12
Validation tickets: 3
Test tickets:       3


## The choice we need validation for

Here is a real development decision. Our word-count classifier (from Lesson 11) scores a ticket by adding up how often each of its words appeared in each category during training. But everyday filler words — "i", "my", "is", "a", "and" — show up everywhere and can pile up enough to drown out the words that actually matter.

So we have **two candidate models**:

- **Candidate A — keep every word**, filler included.
- **Candidate B — remove common filler words** first (a small "stopword" list), so only meaningful words vote.

Which is better? We don't get to just guess — and we're **not allowed** to decide this on the test set. That is precisely the job of the **validation** set: try both, see which wins on data the model didn't learn from, keep the winner.

In [4]:
from collections import Counter  # tallies how many times each word appears

# Common filler words that appear across all categories and carry little meaning
STOPWORDS = {"i", "my", "is", "a", "the", "to", "and", "for", "it", "not",
             "of", "on", "in", "was", "did", "there", "be", "do"}

# Learn one word-count tally per category from the TRAINING tickets only
def train_word_counts(train_df, stopwords=set()):
    counts = {}                                # category name -> its word tally
    for _, row in train_df.iterrows():         # go through training tickets one by one
        words = []                                # this ticket's words, filler removed
        for w in row["tokens"]:                      # check each word in the ticket
            if w not in stopwords:                      # keep it only if it's not filler
                words.append(w)
            else:
                print(f"    dropping filler word: '{w}'")
        print(f"  ticket '{row['text']}' ({row['category']}) -> kept words: {words}")
        if row["category"] not in counts:          # first time seeing this category?
            counts[row["category"]] = Counter()        # start it with an empty tally
            print(f"    first ticket for '{row['category']}', starting a new tally")
        counts[row["category"]].update(words)      # add this ticket's words to that tally
        print(f"    {row['category']} tally is now: {dict(counts[row['category']])}")
    print("\nfinal tallies:", counts, "\n")
    return counts                              # the finished tallies = the learned model

# Guess a category: score each category by summing how often the ticket's words appear in it
def predict(tokens, model, stopwords=set()):
    words = []                          # this ticket's words, filler removed
    for w in tokens:                       # check each word in the ticket
        if w not in stopwords:                # keep it only if it's not filler
            words.append(w)
    print("Words to score:", words)  # show the words that will be scored

    scores = {}                                # category -> total score, built by hand
    for category, counter in model.items():    # one (category, word-tally) pair at a time
        total = 0                                 # running total of matches for this category
        for word in words:                          # check each word in the ticket, one at a time
            match_count = counter.get(word, 0)         # how many times this word appeared here (0 if never seen)
            print(f"    '{word}' seen {match_count}x in {category}")
            total += match_count                        # add it to the running total
        scores[category] = total                   # store this category's finished total
        print(f"  total for {category} = {total}")
    print("Scores:", scores)  # show the scores for each category

    best_category = None    # will hold the highest-scoring category
    best_score = -1          # lower than any real score, so the first category always wins at first
    for category, score in scores.items():   # check each category's score
        if score > best_score:                  # is this the new best one?
            best_score = score
            best_category = category
    print("Winner:", best_category, "\n")  # show the winning category
    return best_category

# Fraction of a table's tickets that the model labels correctly
def accuracy(df, model, stopwords=set()):
    correct = 0                                    # how many guesses were right so far
    for _, row in df.iterrows():                   # check each ticket one by one
        print(f"--- ticket: '{row['text']}' (true = {row['category']}) ---")
        guess = predict(row["tokens"], model, stopwords)  # the model's guess for this ticket
        is_correct = guess == row["category"]                # compare guess to the true answer
        print(f"guess = {guess}   correct = {is_correct}\n")
        if is_correct:
            correct += 1                                        # count it if right
    print(f"Correct Guesses: {correct}")
    print(f"Total Guesses: {len(df)}\n")
    return correct / len(df)

# Train BOTH candidates on the training set only
print("=== training Candidate A (keep all words) ===")
model_keep = train_word_counts(train)                  # Candidate A: keep every word
print("=== training Candidate B (drop filler words) ===")
model_remove = train_word_counts(train, STOPWORDS)     # Candidate B: drop filler words
print("Both candidates trained on", len(train), "tickets.")

=== training Candidate A (keep all words) ===
  ticket 'where is my package it has not arrived yet' (shipping) -> kept words: ['where', 'is', 'my', 'package', 'it', 'has', 'not', 'arrived', 'yet']
    first ticket for 'shipping', starting a new tally
    shipping tally is now: {'where': 1, 'is': 1, 'my': 1, 'package': 1, 'it': 1, 'has': 1, 'not': 1, 'arrived': 1, 'yet': 1}
  ticket 'my delivery is late and tracking has not updated' (shipping) -> kept words: ['my', 'delivery', 'is', 'late', 'and', 'tracking', 'has', 'not', 'updated']
    shipping tally is now: {'where': 1, 'is': 2, 'my': 2, 'package': 1, 'it': 1, 'has': 2, 'not': 2, 'arrived': 1, 'yet': 1, 'delivery': 1, 'late': 1, 'and': 1, 'tracking': 1, 'updated': 1}
  ticket 'when will my order ship out' (shipping) -> kept words: ['when', 'will', 'my', 'order', 'ship', 'out']
    shipping tally is now: {'where': 1, 'is': 2, 'my': 3, 'package': 1, 'it': 1, 'has': 2, 'not': 2, 'arrived': 1, 'yet': 1, 'delivery': 1, 'late': 1, 'and': 1

### Now choose — on the validation set

We score each candidate on the 3 validation tickets (which neither model learned from) and keep whichever does better. This is the *only* place we're allowed to compare and choose.

In [5]:
print("############ Evaluating Candidate A (keep all words) on VALIDATION ############")
acc_keep = accuracy(validation, model_keep)

print("############ Evaluating Candidate B (remove filler words) on VALIDATION ############")
acc_remove = accuracy(validation, model_remove, STOPWORDS)

print(f"Candidate A (keep all words)   validation accuracy: {acc_keep:.2f}")
print(f"Candidate B (remove filler)    validation accuracy: {acc_remove:.2f}")

# Pick the winner based ONLY on validation
if acc_remove >= acc_keep:
    print(f"B's accuracy ({acc_remove:.2f}) >= A's ({acc_keep:.2f}) -> choosing Candidate B")
    chosen_model, chosen_stopwords, chosen_name = model_remove, STOPWORDS, "Candidate B (remove filler)"
else:
    print(f"B's accuracy ({acc_remove:.2f}) < A's ({acc_keep:.2f}) -> choosing Candidate A")
    chosen_model, chosen_stopwords, chosen_name = model_keep, set(), "Candidate A (keep all words)"

print("Chosen model:", chosen_name)

############ Evaluating Candidate A (keep all words) on VALIDATION ############
--- ticket: 'i think my order is late' (true = shipping) ---
Words to score: ['i', 'think', 'my', 'order', 'is', 'late']
    'i' seen 0x in shipping
    'think' seen 0x in shipping
    'my' seen 4x in shipping
    'order' seen 1x in shipping
    'is' seen 3x in shipping
    'late' seen 1x in shipping
  total for shipping = 9
    'i' seen 4x in billing
    'think' seen 0x in billing
    'my' seen 3x in billing
    'order' seen 1x in billing
    'is' seen 1x in billing
    'late' seen 0x in billing
  total for billing = 9
    'i' seen 4x in account
    'think' seen 0x in account
    'my' seen 5x in account
    'order' seen 0x in account
    'is' seen 1x in account
    'late' seen 0x in account
  total for account = 10
Scores: {'shipping': 9, 'billing': 9, 'account': 10}
Winner: account 

guess = account   correct = False

--- ticket: 'there is a wrong charge on my card again' (true = billing) ---
Words to sco

### Why did keeping every word lose?

Look at the validation ticket **"i think my order is late"** — it's really about *shipping*. But "i", "my", and "is" happen to pile up under **account** and **billing** in training, so Candidate A adds them up and wrongly picks account. Candidate B throws those filler words away first, so only "order" and "late" get to vote — and they point at shipping. That single ticket is the difference, and the validation set is what let us *see* it before shipping anything.

## The final judge — the test set, touched once

We've now made our choice. The model is finished. Only now do we unlock the **test** set — the 3 tickets that never touched training *or* our tuning decision — and score the chosen model on it, one time. This number is our honest estimate of how the assistant would do on real, unseen tickets tomorrow.

In [6]:
test_accuracy = accuracy(test, chosen_model, chosen_stopwords)  # the ONE look at the test set

print(f"Final test accuracy of {chosen_name}: {test_accuracy:.2f}")

# Show each test ticket's prediction so we can see it wasn't luck
for _, row in test.iterrows():
    guess = predict(row["tokens"], chosen_model, chosen_stopwords)  # the model's label
    mark = "OK " if guess == row["category"] else "XX "             # right or wrong
    print(f"{mark} true={row['category']:<8} guess={guess:<8} | {row['text']}")

--- ticket: 'my package has not arrived and delivery is late' (true = shipping) ---
Words to score: ['package', 'has', 'arrived', 'delivery', 'late']
    'package' seen 2x in shipping
    'has' seen 3x in shipping
    'arrived' seen 2x in shipping
    'delivery' seen 1x in shipping
    'late' seen 1x in shipping
  total for shipping = 9
    'package' seen 0x in billing
    'has' seen 0x in billing
    'arrived' seen 0x in billing
    'delivery' seen 0x in billing
    'late' seen 0x in billing
  total for billing = 0
    'package' seen 0x in account
    'has' seen 0x in account
    'arrived' seen 0x in account
    'delivery' seen 0x in account
    'late' seen 0x in account
  total for account = 0
Scores: {'shipping': 9, 'billing': 0, 'account': 0}
Winner: shipping 

guess = shipping   correct = True

--- ticket: 'i was charged twice and need a refund' (true = billing) ---
Words to score: ['charged', 'twice', 'need', 'refund']
    'charged' seen 0x in shipping
    'twice' seen 0x in ship

In [8]:
print("############ FINAL TEST SET EVALUATION (touched exactly once) ############")
test_accuracy = accuracy(test, chosen_model, chosen_stopwords)  # the ONE look at the test set

print(f"Final test accuracy of {chosen_name}: {test_accuracy:.2f}\n")

# Show each test ticket's prediction so we can see it wasn't luck
print("############ Per-ticket results ############")
for _, row in test.iterrows():
    guess = predict(row["tokens"], chosen_model, chosen_stopwords)  # the model's label
    if guess == row["category"]:   # did the model get it right?
        mark = "OK "
    else:
        mark = "XX "
    print(f"{mark} true={row['category']:<8} guess={guess:<8} | {row['text']}")

############ FINAL TEST SET EVALUATION (touched exactly once) ############
--- ticket: 'my package has not arrived and delivery is late' (true = shipping) ---
Words to score: ['package', 'has', 'arrived', 'delivery', 'late']
    'package' seen 2x in shipping
    'has' seen 3x in shipping
    'arrived' seen 2x in shipping
    'delivery' seen 1x in shipping
    'late' seen 1x in shipping
  total for shipping = 9
    'package' seen 0x in billing
    'has' seen 0x in billing
    'arrived' seen 0x in billing
    'delivery' seen 0x in billing
    'late' seen 0x in billing
  total for billing = 0
    'package' seen 0x in account
    'has' seen 0x in account
    'arrived' seen 0x in account
    'delivery' seen 0x in account
    'late' seen 0x in account
  total for account = 0
Scores: {'shipping': 9, 'billing': 0, 'account': 0}
Winner: shipping 

guess = shipping   correct = True

--- ticket: 'i was charged twice and need a refund' (true = billing) ---
Words to score: ['charged', 'twice', 'nee